# Разработка модели машинного обучения

## Импортирование библиотек

In [6]:
# для работы с датафреймами
import pandas as pd

# для визуализации результатов
import matplotlib.pyplot as plt

# для работы с массивами
import numpy as np

# для преобразования текста
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize

# для работы со строками
import string

# вспомогательные функции
from function import *

# для работы с датасетами
from datasets import Dataset, DatasetDict

# для обработки текста
from pymystem3 import Mystem

# токенизатор и модель
from transformers import T5Tokenizer, T5ForConditionalGeneration
# аргументы для обучения, и трейнер
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
# коллатор
from transformers import DataCollatorForSeq2Seq

# основной модуль для нейронных сетей
import torch

# модуль с метрикой оценивания
import evaluate


Определяю устройство, на котором будут производиться вычисления:

In [7]:
# получаю девайс
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# вывожу девайс
print(device)

cuda


## Загрузка данных

In [8]:
# загружаю тренировочную выборку
train_dataset = Dataset.from_parquet("Dataset/train.parquet")
# валидационную выборку
validation_dataset = Dataset.from_parquet("Dataset/validation.parquet")
# тестовую выборку
test_dataset = Dataset.from_parquet("Dataset/test.parquet")
# загружаю тренировочную выборку
train_dataset = Dataset.from_parquet("Dataset/train.parquet")
# валидационную выборку
validation_dataset = Dataset.from_parquet("Dataset/validation.parquet")
# тестовую выборку
test_dataset = Dataset.from_parquet("Dataset/test.parquet")

# объединяю все выборки в объект Dataset
# dataset = DatasetDict({
# объединяю все выборки в объект Dataset
dataset = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset
})
# вывожу структуру получившегося набора данных
dataset

DatasetDict({
    train: Dataset({
        features: ['text_path', 'annotation_path', 'tags_path', 'text', 'summary', 'tag', 'text_all_symb', 'summary_all_symb', 'tag_all_symb', 'text_clean', 'summary_clean', 'tag_clean', 'text_words', 'summary_words', 'tag_words', 'id', 'processed_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 329
    })
    validation: Dataset({
        features: ['text_path', 'annotation_path', 'tags_path', 'text', 'summary', 'tag', 'text_all_symb', 'summary_all_symb', 'tag_all_symb', 'text_clean', 'summary_clean', 'tag_clean', 'text_words', 'summary_words', 'tag_words', 'id', 'processed_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 41
    })
    test: Dataset({
        features: ['text_path', 'annotation_path', 'tags_path', 'text', 'summary', 'tag', 'text_all_symb', 'summary_all_symb', 'tag_all_symb', 'text_clean', 'summary_clean', 'tag_clean', 'text_words', 'summary_words', 'tag_words', 'id', 'processed_text', 'input_ids', 

## Подбор алгоритма обучения

Блаблабла чоооо швепс эщкере

### Алгоритм обучения

Передо мной стоит задача `суммаризации` текста. Это значит, что надо реализовать нейронную сеть, которая будет находить и выписывать краткое содержание текста. Для этого я буду использовать предобученную модель `"sarahai/ruT5-base-summarizer"`, которую настрою работать на своих данных

### Метрики для суммаризации текста

Для суммаризации одной из наиболее часто используемых метрик является оценка `ROUGE` (сокращение от `Recall-Oriented Understudy for Gisting Evaluation`). Основная идея этой метрики заключается в сравнении сгенерированного текста с набором эталонных текстов, которые обычно создаются людьми. Чтобы сделать ее более точной, предположим, что мы хотим сравнить следующие две строчки:

In [9]:
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"

Одним из способов их сравнения может быть подсчет `количества перекрывающихся слов`, которых в данном случае будет 6. Однако это несколько грубовато, поэтому вместо этого `ROUGE` основывается на вычислении оценок `precision` и `recall` для перекрытия

Для `ROUGE` `recall` измеряет, насколько эталонное резюме соответствует `сгенерированному`. Если мы просто сравниваем слова, `recall` можно рассчитать по следующей формуле:

$$\text{Recall} = \frac{\text{Number of overlapping words}}{\text{Total number of words in reference summary}}$$

Для нашего простого примера выше эта формула дает идеальный `recall` 6/6 = `1`; то есть все слова в эталонном тексте были получены моделью. Это может показаться замечательным, но представьте, если бы сгенерированный нами текст был “I really really loved reading the Hunger Games all night”. Это тоже дало бы идеальный `recall`, но, возможно, было бы хуже, поскольку было бы многословным. Чтобы справиться с этими сценариями, мы также вычисляем `precision`, которая в контексте `ROUGE` измеряет, насколько `сгенерированное` резюме было `релевантным`:

$$\text{Precision} = \frac{\text{Number of overlapping words}}{\text{Total number of words in generated summary}}$$

Если применить это к нашему подробному тексту, то `precision` составит 6/10 = 0,6, что значительно хуже, чем `precision` 6/7 = 0,86, полученная при использовании более короткого текста. На практике обычно вычисляют и `precision`, и `recall`, а затем `F1-score` (среднее гармоническое из `precision` и `recall`)

Загружаю метрику `ROUGE`:

In [10]:
# загружаю используемую метрику
rouge_score = evaluate.load('rouge')

Затем мы можем использовать функцию `rouge_score.compute()`, чтобы рассчитать все метрики сразу:

In [11]:
# считаем метрики примеров
scores = rouge_score.compute(
    predictions=[generated_summary],
    references=[reference_summary]
)
# вывожу получившиеся метрики
scores

{'rouge1': 0.923076923076923,
 'rouge2': 0.7272727272727272,
 'rougeL': 0.923076923076923,
 'rougeLsum': 0.923076923076923}

## Работа с нейронной сетью

#### Модель суммаризации

Для начала, надо инициализировать саму `модель` и ее `токенизатор`:

In [7]:
# имя модели
model_name = "sarahai/ruT5-base-summarizer"
# инициализируем модель
model = T5ForConditionalGeneration.from_pretrained(model_name)
# инициализирую токенизатор
tokenizer = T5Tokenizer.from_pretrained(model_name)

Теперь, надо назначить вычислительное устройство для модели (его я определил выше):

In [8]:
# назначаю устройство и вывожу архитектуру модели
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

Дообучение `ruT5` с `API` `Trainer`

In [9]:
# назначаю кол-во батчей
batch_size = 8
# кол-во эпох
epochs = 30
# функция потерь
logging_steps = len(dataset['train']) // batch_size
name = model_name.split('/')[-1]
print(name)

ruT5-base-summarizer


Теперь, нужно создать объект `Seq2SeqTrainingArguments`, чтобы заполнить гиперпараметры модели:

In [10]:
# объект с гиперпараметрами
args = Seq2SeqTrainingArguments(
    output_dir='model-4-summary',
    overwrite_output_dir=True,
    eval_strategy='epoch',                     # оценка после каждой эпохи
    per_device_train_batch_size=batch_size,     # кол-во тренировочных батчей
    per_device_eval_batch_size=batch_size,      # кол-во батчей для оценки
    gradient_accumulation_steps=2,              # кол-во шагов накопления градиента до обновления
    torch_empty_cache_steps=4,                  # oчистка кэша GPU через каждые 4 шага
    learning_rate=1e-4,                         # скорость обучения
    num_train_epochs=epochs,                    # кол-во эпох
    logging_steps=logging_steps,                # частота логов
    seed=42,                                    # сид для воспроизводимости результатов
    fp16=True,                                  # bbспользование mixed precision для ускорения обучения
    weight_decay=0.01,                          # отложенные весаL2-регуляризация для предотвращения переобучения
    optim='adamw_torch',                        # оптимизатор
    # report_to="tensorboard",

)

Следующее, что нужно сделать, это предоставить тренеру функцию `compute_metrics()`, чтобы оценить нашу модель во время обучения. Для суммаризации это немного сложнее, чем просто вызвать `rouge_score.compute()` для прогнозов модели, поскольку нужно декодировать выводы и метки в текст, прежде чем вычислить оценку `ROUGE`. Следующая функция делает именно это, а также использует функцию `sent_tokenize()` из `nltk` для разделения предложений резюме символом новой строки

In [11]:
# функция для вычисления метрик
def compute_metrics(eval_pred):
    # получаем предсказания и их метки
    predictions, labels = eval_pred
    # print(f"Predictions {predictions}, shape: {predictions[0].shape}, type: {type(predictions)}")

    # print(f"Labels {labels}, shape: {labels.shape}, type: {type(labels)}")
    # Если predictions — это кортеж, берем первый элемент (логи)
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Преобразуем логи в индексы токенов с помощью argmax
    predicted_token_ids = np.argmax(predictions, axis=-1)
    # декодируем сгенерированные суммаризации в текст
    decoded_preds = tokenizer.batch_decode(predicted_token_ids, skip_special_tokens=True)
    # заменяем -100 в метках, тк декодировать их нельзя
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # декодируем эталонные изложения 
    decoded_summary = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # ROUGE ожидает символ новой строки после каждого предложения
    decoded_preds = ['\n'.join(sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_summary = ['\n'.join(sent_tokenize(label.strip())) for label in decoded_summary]
    
    # вычисляем метрики ROUGE
    result = rouge_score.compute(
        predictions=decoded_preds,
        references=decoded_summary,
        use_stemmer=True            # проверить и с ним, и без него
    )

    # получаем оценки
    result = {k: v*100 for k, v in result.items()}
    return {k: round(v,4) for k,v in result.items()}

Также, для обучения модели необходим `коллатор`, который будет сдвигать метки на 1 каждый шаг

In [12]:
# инициализируем коллатор
collator = DataCollatorForSeq2Seq(tokenizer, model)

Получаем датасет для обучения модели

In [13]:
# убираем лишние колонки
train_data = dataset.select_columns(['input_ids', 'attention_mask', 'labels'])
# выводим датасет
train_data

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 329
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 41
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 42
    })
})

In [14]:
type(train_data['train']['input_ids'][0])

list

Теперь, надо создать объект `Trainer` для запуска обучения модели

In [15]:
# объект trainer 
trainer = Seq2SeqTrainer(
    model,                                  # модель
    args,                                   # тренировочные аргументы
    train_dataset=train_data['train'],      # датасет для обучения модели
    eval_dataset=train_data['validation'],  # датасет для оценки модели
    data_collator=collator,                 # коллатор
    processing_class=tokenizer,             # токенизатор
    compute_metrics=compute_metrics         # функция для вычисления метрик
)

Очищаю кэш видеокарты, чтобы разгрузить память

In [16]:
torch.cuda.empty_cache()

После этого, можно начать обучение модели

In [17]:
# запуск обучения модели
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,1.661759,11.037000,2.570900,11.012300,11.070800
2,1.826200,1.642799,11.373600,2.439000,11.406800,11.441800
3,1.826200,1.636124,9.957300,2.623800,9.823600,9.897200
4,1.424400,1.635471,10.178300,2.623800,10.250300,10.183900
5,1.424400,1.657753,11.335700,2.702700,11.405900,11.400800
6,1.194900,1.663106,11.655000,2.695800,11.722200,11.665400
7,1.194900,1.701361,13.588100,2.664200,13.450500,13.616400
8,1.029700,1.762112,12.066900,2.439000,11.648200,12.085700
9,1.029700,1.774765,14.967100,3.353700,14.342600,14.676300
10,0.879100,1.787301,13.137000,3.515700,12.565900,12.880200


TrainOutput(global_step=630, training_loss=0.7540824451143779, metrics={'train_runtime': 5274.217, 'train_samples_per_second': 1.871, 'train_steps_per_second': 0.119, 'total_flos': 6010414379827200.0, 'train_loss': 0.7540824451143779, 'epoch': 30.0})

Сохраняю полученную модель

In [18]:
# Сохранение модели
model.save_pretrained("./saved_model")

# Сохранение токенизатора
tokenizer.save_pretrained("./saved_model")

('./saved_model\\tokenizer_config.json',
 './saved_model\\special_tokens_map.json',
 './saved_model\\spiece.model',
 './saved_model\\added_tokens.json')

Загружаю модель из файла

In [13]:
# загрузка модели
model = T5ForConditionalGeneration.from_pretrained("./saved_model")

# загрузка токенизатора
tokenizer = T5Tokenizer.from_pretrained("./saved_model")

Далее, опять назначаю устройство вычисление для модели

In [14]:
# привожу модель к устройству
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

Тестирую полученную модель суммаризации:

In [23]:
# индекс строки с примером из датасета
index = 4
# оригинальные текст
orig_text = dataset['validation']['text'][index]
# оригинальная суммаризация
summary = dataset['validation']['summary'][index]
# выводим суммаризацию
print(summary)

Анализируется феномен современного цифрового общества - новый технологический уклад, интегрирующий инфокоммуникативные технологии, радикально и стремительно преобразует ландшафт человеческой телесности, повседневности и духовности. Рассмотрены точки зрения ряда российских исследователей по взаимодействию философии, с одной стороны, и цифровых технологий, с другой, в части формирования системы гуманитарных знаний по цифровой философии и цифровому человеку. Определены смысл и роль философской рефлексии в меняющемся мире в прояснении и систематизации спорных вопросов о статусе человека, в сохранении его подлинно «человеческой» сущности.


Используя реализованную в файле `function.py` функцию `get_summary`, попробую получить суммаризацию данного текста:

In [25]:
# вывожу суммаризацию
generated_sum = get_summary(text=orig_text, 
            tokenizer=tokenizer, 
            model=model, 
            device=device, 
            show_output=True)    # включаю автоматический вывод результатов

>> Original Text: Человечество в очередной раз входит в
эпоху глобальных перемен, на этот раз в эпоху
цифровизации. Хотя каждое поколение, конечно же, может заявить о своей эпохе Великих
перемен и/или потрясений, да и не об одной.
В современном мире, где огромную роль
играет формирующаяся цифровая культура, все
сферы социального бытия и социальной практики, многие области знания трансформируются под воздействием инфокоммуникативных
технологий.
Простое использование цифровых технологий в гуманитарных науках, в первую очередь в философии, вызывает сложности не
только методологического и методического характера, но и неясности в самом определении
области применения.
Исследователь Е. Елькина (2020) подчеркивает, что «Технологическая гонка, знаменующая переход ведущих экономических держав
в шестой технологический уклад, породила
большой поток терминов, в которых «дигитальность» рассматривается как маркер изменений
предметных областей, где эти технологии используются («цифровая экономика», «

пробую посчитать метрику `ROUGE` на всем валидационном датасете

In [36]:
rouge1 = []             # список для метрики rouge1
rouge2 = []             # список для метрики rouge2
rougeL = []             # список для метрики rougeL
rougeLsum = []          # список для метрики rougeLsum
# прохожусь по каждому текста из валидационного набора
for i in tqdm(range(len(dataset['validation'])), desc='Вычисление метрик..', unit='sample'):
    # оригинальные текст
    orig_text = dataset['validation']['text'][i]
    # оригинальная суммаризация
    summary = dataset['validation']['summary'][i]
    # генерирую суммаризацию для текста
    generated_sum = get_summary(text=orig_text, 
            tokenizer=tokenizer, 
            model=model, 
            device=device)    
    
    # вычисляю метрики
    result = rouge_score.compute(
        predictions=[generated_sum],
        references=[summary],
    )

    # добавляю метрики в списки
    rouge1.append(result['rouge1'])
    rouge2.append(result['rouge2'])
    rougeL.append(result['rougeL'])
    rougeLsum.append(result['rougeLsum'])
# вывожу значения средних метрик на валидационном датасете
print('Метрики на валидационном датасете:')  
print(f'rouge1: {sum(rouge1)/len(rouge1)}')
print(f'rouge2: {sum(rouge2)/len(rouge2)}')
print(f'rougeL: {sum(rougeL)/len(rougeL)}')
print(f'rougeLsum: {sum(rougeLsum)/len(rougeLsum)}')

Вычисление метрик..: 100%|██████████| 41/41 [06:44<00:00,  9.87s/sample]

Метрики на валидационном датасете:
rouge1: 0.11998457533982454
rouge2: 0.018827556696619598
rougeL: 0.10945242456377133
rougeLsum: 0.10945242456377133


Хоть по метрикам модель справляется плохо, после тестирования могу сказать что суммаризация стабильная и понятная

#### Модель для сравнения

Для начала, надо импортировать нужные модули

In [38]:
# модель извлечеия эмбеддингов
from gensim.models import Word2Vec

# модуль для nlp
import gensim

# обработка текста и предложений
from nltk.tokenize import sent_tokenize, word_tokenize

Получу случайный текст из датасета, для тестирования `Word2Vec` извлечения эмбеддингов

In [59]:
# текст
text = get_input(dataset['train']['text'][52])

In [64]:
len(dataset['train']['text'])

329

In [123]:
data = []
# проходимся по каждому тексту в датасете
for i in tqdm(range(len(dataset['train']['processed_text'])), desc='Собираем данные..', unit='text'):
    # извлекаем текст
    text = dataset['train']['processed_text'][i]
    # проходимся по токенизированным предложениям в тексте
    for sentence in sent_tokenize(text.replace('\n', ' ')):
        # список для сохранения токенизированных слов
        temp = []
        # токенизируем слова в предложениях
        for word in word_tokenize(sentence):
            # добавляем слово в временный список
            temp.append(word.lower())
        # добавляем предложение к данным
        data.append(temp)
# выводим колво предложений
print(len(data))


Собираем данные..: 100%|██████████| 329/329 [00:04<00:00, 67.17text/s]

17083


Создаем объект `CBOW` модели

In [124]:
# создаем CBOW модель
model_cbow = gensim.models.Word2Vec(data, min_count=1, vector_size=100, window=5)

In [125]:
# Print results
word1 = 'плохой'
word2 = 'хороший'
print(f"Cosine similarity between '{word1}' " +
      f"and '{word2}' - CBOW : ",
      model_cbow.wv.similarity(word1, word2))

word3 = 'плохой'
word4 = 'хороший'
print(f"Cosine similarity between '{word3}' " +
      f"and '{word4}' - CBOW : ",
      model_cbow.wv.similarity(word3, word4))

Cosine similarity between 'плохой' and 'хороший' - CBOW :  0.9775187
Cosine similarity between 'плохой' and 'хороший' - CBOW :  0.9775187


In [126]:
# Находим слова, максимально непохожие на "кошка"
negative_words = model_cbow.wv.most_similar(positive=[], negative=['плохой'], topn=5)
print(negative_words)

[('кружок', 0.8812046051025391), ('искаж\x02ниям', 0.8809285163879395), ('views', 0.8641453385353088), ('heretofore', 0.8283572793006897), ('summon', 0.7393378019332886)]


Создаем `skip-gram` модель

In [127]:
# создаем skip-gram модель
model_skipgram = gensim.models.Word2Vec(data, min_count=1, vector_size=100, window=5, sg=1)

In [128]:
# Print results
word1 = 'человек'
word2 = 'женщина'
print(f"Cosine similarity between '{word1}' " +
      f"and '{word2}' - CBOW : ",
      model_skipgram.wv.similarity(word1, word2))

word3 = 'человек'
word4 = 'мужчина'
print(f"Cosine similarity between '{word3}' " +
      f"and '{word4}' - CBOW : ",
      model_skipgram.wv.similarity(word3, word4))

Cosine similarity between 'человек' and 'женщина' - CBOW :  0.75798583
Cosine similarity between 'человек' and 'мужчина' - CBOW :  0.66121817


In [129]:
# Находим слова, максимально непохожие на "кошка"
negative_words = model_skipgram.wv.most_similar(positive=['женщина'], negative=[], topn=5)
print(negative_words)

[('мироздание', 0.9827036261558533), ('родитель', 0.9826143980026245), ('облик', 0.9826050996780396), ('невидимый', 0.9825860261917114), ('раса', 0.982400119304657)]


In [152]:
dataset['train']['text'][0]

'При разработке проекта одним из важнейших этапов является\nсоставление сметной документации. Строительная смета – это основной\nдокумент, в котором заранее подсчитаны все затраты, с которыми заказчик\nстолкнется в процессе проведения строительных и ремонтных работ. В строительной смете учитываются затраты на отделочные, подготовительные,\nкровельные, земляные, монтажные, каменные работы, а также санитарнотехнические и специальные виды работы.\nВ случае возникновения ряда спорных вопросов заказчик может\nобратиться в арбитражный суд с иском о некачественном выполнении работ,\nзавышении стоимости выполненных строительных работ и несоответстствии\nвида строительной продукции той, что была указана в проекте. Судебная\nстроительно-техническая экспертиза назначается в том случае, когда суду,\nследствию или органу дознания не достаточно знаний в строительной\nобласти. В рамках арбитражного делопроизводства правовой основой\nсудебной экспертизы является Конституция Российской Федерации и\nАрб

Пробую использовать `Doc2Vec` 

In [166]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [132]:
# создаем список для хранения данных
data = []
# проходимся по каждому тексту в датасете
for i in tqdm(range(len(dataset['train']['summary'])), desc='Собираем данные..', unit='text'):
    # извлекаем текст
    data.append(dataset['train']['summary'][i])
    

Собираем данные..: 100%|██████████| 329/329 [00:00<00:00, 965.92text/s]


In [133]:
# токенизируем данные
tokenized_data = [word_tokenize(summary.lower()) for summary in data]

In [134]:
# создаем объект TaggedDocument
tagged_data = [TaggedDocument(words=words, tags=[str(idx)])
               for idx, words in enumerate(tokenized_data)]


In [140]:
# тренируем модель Doc2Vec
model = Doc2Vec(vector_size=100, window=2, min_count=1, workers=4, epochs=1000)
model.build_vocab(tagged_data)
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)

In [178]:
# сохранение
model.save('doc2vec.model')

In [179]:
# загрузка
model = Doc2Vec.load('doc2vec.model')

In [203]:
new_summary1 = dataset['test']['summary'][4]
new_summary2 = dataset['test']['summary'][1]

In [204]:
# вычисляем эмбеддинг
inferred_vector1 = model.infer_vector(word_tokenize(new_summary1.lower())).reshape(1,-1)
inferred_vector2 = model.infer_vector(word_tokenize(new_summary2.lower())).reshape(1,-1)

In [212]:
vectors = [inferred_vector1, inferred_vector2]
texts = [new_summary1, new_summary2]

In [213]:
print(inferred_vector2.shape)
print(inferred_vector2.reshape(1,-1).shape)

(1, 100)
(1, 100)


In [206]:
type(model) == Doc2Vec

True

In [207]:
# функция для сравнения двух текстов
def get_similarity(text1: str, text2: str, model: str | Doc2Vec):
    '''
    Функция для получения схожести двух текстов
    ===
        Args:
            - text1 (str): первый текст в строковом формате
            - text2 (str): второй текст в строковом формате
            - model (str|Doc2Vec): модель в формате doc2vec объекта 
                                  либо путь к ней в строковом формате

        Returns:
            - float: сходство между текстами (от 1 до -1)
    '''
    # проверка формата текста 1
    if not isinstance(text1, str):
        raise TypeError(f'text1 должен быть в строковом формате, а не {type(text1)}')
    # проверка формата текста 2
    if not isinstance(text1, str):
        raise TypeError(f'text1 должен быть в строковом формате, а не {type(text1)}')
    # проверка формата модели
    if not isinstance(model, (str, Doc2Vec)):
        raise TypeError(f'model должна быть в формате Doc2Vec, либо в виде строкового путя, а не {type(model)}')
    # если модель в виде путя
    if type(model) == str:
        # проверяем, что файл существует
        if os.path.exists(model):
            # пробуем загрузить модель через try: except
            try:
                # загружаем модель из файла
                model = Doc2Vec.load(model)
            # если не получилось загрузить, выводим ошибку
            except Exception as e:
                raise ValueError(f'Ошибка! не удалось загрузить модель, проверьте ваш файл!\n{e}')
        # если путь не существует:
        else:
            # выводим ошибку
            raise ValueError('Ошибка! Файла с моделью не существует, проверьте правильность написания!')
    
    # вычисляем эмбеддинг
    inferred_vector1 = model.infer_vector(word_tokenize(text1.lower())).reshape(1,-1)
    inferred_vector2 = model.infer_vector(word_tokenize(text2.lower())).reshape(1,-1)
    # получаем сходство
    return cosine_similarity(inferred_vector1, inferred_vector2).item()

In [208]:
similarity = get_similarity(new_summary1, new_summary2, model)

Тестирование предобученной модели 

In [216]:
# импортирование библиотек
from scipy.spatial import distance
from sentence_transformers import SentenceTransformer

Загружаем модель

In [217]:
model_similarity = SentenceTransformer('all-MiniLM-L6-v2')

c:\Users\user1\NLP\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user1\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [221]:
text1_vec = model_similarity.encode(new_summary1).reshape(1,-1)
text2_vec = model_similarity.encode(new_summary2).reshape(1,-1)

In [223]:
from sklearn.metrics.pairwise import cosine_similarity

In [224]:
cosine_similarity(text1_vec, text2_vec).item()

0.8189477920532227

## Анализ эффективности модели

## Оптимизация

## Разработка программного продукта